In [110]:
import pandas as pd

df_train = pd.read_csv('train_data.csv')
df_test = pd.read_csv('test_data.csv')

df_test.head()

,datapointID,Word,Similar
0,1,beverage,pasionat
1,2,analyze,adopta
2,3,film,audio
3,4,dome,vultur
4,5,enlarge,atenție


In [111]:
#subtask 1
cnt = 0

for en_word in df_test.Word:
    for ro_word in df_test.Similar:
        if en_word == ro_word:
            cnt += 1

subtask1 = cnt


df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": [0],
    "answer": subtask1
})

df_1

,subtaskID,datapointID,answer
0,1,0,124


In [112]:
import numpy as np

def load_vec(path, limit=50000): #asta fac ca nu imi merge gensimul
    vectors = {}
    with open(path, encoding='utf-8') as f:
        next(f)  # skip header
        for i, line in enumerate(f):
            if i >= limit:
                break
            parts = line.rstrip().split(' ')
            word = parts[0]
            vec = np.array(parts[1:], dtype=float)
            vectors[word] = vec
    return vectors

model_en = load_vec('./fasttext/wiki.en.vec', 50000)
model_ro = load_vec('./fasttext/wiki.ro.vec', 50000)

In [113]:
df_train.head()

import re

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df_train.english = df_train.english.apply(preprocess)
df_train.romanian = df_train.romanian.apply(preprocess)

df_train.head()

,english,romanian
0,if you add three and four you get seven,dacă aduni trei cu patru obții șapte
1,at first i didnt like it but it gradually beca...,la început nu mia plăcut dar treptat a devenit...
2,he made me love jazz,el ma făcut să îndrăgesc jazzul
3,what time will you leave,la ce oră vei pleca
4,she went with him to boston,ea a plecat cu el la boston


In [114]:
def get_en_mean(sentence):
    words = sentence.split()
    vecs = []
    for word in words:
        if word in model_en:
            vec = model_en[word]
            vecs.append(vec)
        else:
            vecs.append(np.zeros(300))
    return np.mean(vecs, axis=0)

def get_ro_mean(sentence):
    words = sentence.split()
    vecs = []
    for word in words:
        if word in model_ro:
            vec = model_ro[word]
            vecs.append(vec)
        else:
            vecs.append(np.zeros(300))
    return np.mean(vecs, axis=0)

embs_en = []
embs_ro = []
for _, row in df_train.iterrows():
    embs_en.append(get_en_mean(row.english))
    embs_ro.append(get_ro_mean(row.romanian))

embs_en = np.array(embs_en)
embs_ro = np.array(embs_ro)

from sklearn.preprocessing import normalize
embs_en = normalize(embs_en)
embs_ro = normalize(embs_ro)

embs_en.shape

(15182, 300)

In [115]:
from sklearn.linear_model import Ridge
model = Ridge()
model.fit(embs_en, embs_ro)

model_en["beverage"][:10]

array([-0.0899, -0.0638,  0.1536,  0.0827,  0.0923, -0.0181,  0.1305,
       -0.1299, -0.1338, -0.1939])

In [116]:
dim = 300

X_q = []
X_c = []

for word in df_test.Word:
    X_q.append(model_en.get(word, np.zeros(dim)))

for word in df_test.Similar:
    X_c.append(model_ro.get(word, np.zeros(dim)))

In [117]:
import numpy as np

X_q = np.array(X_q)
X_c = np.array(X_c)

X_q = model.predict(X_q)

X_q = normalize(X_q)
X_c = normalize(X_c)

X_q.shape

(1115, 300)

In [118]:
from sklearn.metrics.pairwise import cosine_similarity

word = X_q[1].reshape(1, -1)
cand = X_c[317].reshape(1, -1)

cosine_similarity(word, cand)[0][0]

np.float64(0.15516325567688607)

In [119]:
from tqdm import tqdm

inds = []

for query in tqdm(X_q):
    word = query.reshape(1, -1)
    ind = np.argmax(cosine_similarity(word, X_c)[0, :])
    inds.append(ind)

inds[:5]

100%|█████████████████████████████████████████████████████████████████████████████| 1115/1115 [00:03<00:00, 341.10it/s]


[np.int64(43), np.int64(876), np.int64(1002), np.int64(1001), np.int64(185)]

In [120]:
preds = []

for i in inds:
    word = df_test.Similar.iloc[int(i)]
    preds.append(word)

preds[:5]

['băutură', 'analiza', 'film', 'cupolă', 'crea']

In [121]:
df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.datapointID,
    "answer": preds
})

df_2.head()

,subtaskID,datapointID,answer
0,2,1,băutură
1,2,2,analiza
2,2,3,film
3,2,4,cupolă
4,2,5,crea


In [122]:
submission = pd.concat([df_1, df_2])
submission

,subtaskID,datapointID,answer
0,1,0,124
0,2,1,băutură
1,2,2,analiza
2,2,3,film
3,2,4,cupolă
...,...,...,...
1110,2,1111,defini
1111,2,1112,chiar
1112,2,1113,umple
1113,2,1114,lângă


In [123]:
submission.to_csv("submission.csv", index=False)